# First-token gap vs full-answer delta

**The question.** For tulu, mean `delta` (eval.py's length-normalised full-answer
log-prob gap, positive = favours the CONTEXT answer) barely moves across the decline:
peak `+1.177` -> trough `+1.093`, with only 54% of items moving toward the parametric
answer -- essentially chance. Yet R_ctx over those same checkpoints drops 0.955 -> 0.776.
So at the trough the model still *scores* the contextual answer higher, but its greedy
output increasingly *states* the parametric one.

**The hypothesis.** `delta` averages over all answer tokens, but greedy generation
commits at the **first token**. An answer can win on average and lose at token 1. If the
inversion lives in first-token commitment, then the first-token gap should track the
R_ctx trajectory (drop peak->trough, recover) while `delta` does not.

**The measurement.** For every item that passed the Layer-0 filter, at each phase
checkpoint, record both quantities on the *same* prompt:

- `delta`      -- `eval.py`'s own `method_logprob` (unchanged, reused directly)
- `first_gap`  -- `logit(cf_first_token) - logit(par_first_token)` at the last prompt
                  position, same sign convention

No judge, no API, forward passes only. ~400 items x 5 checkpoints.

### Repo + Drive + checkpoints

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import yaml, shutil

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))
STAGE_DIR = "/content/staged_checkpoints"

for run, run_cfg in cfg["runs"].items():
    for phase, ckpt_name in run_cfg["phases"].items():
        if ckpt_name is None:
            continue
        dst = os.path.join(STAGE_DIR, run, ckpt_name)
        if os.path.isdir(dst):
            print("already staged:", run, ckpt_name)
            continue
        print("staging", run, phase, ckpt_name)
        shutil.copytree(os.path.join(run_cfg["drive_checkpoint_dir"], ckpt_name), dst)

def phase_path(run, phase):
    c = cfg["runs"][run]["phases"][phase]
    return None if c is None else os.path.join(STAGE_DIR, run, c)

### Model + tokenizer (chat template is load-bearing -- see the assert)

In [ ]:
!pip install -q peft accelerate

import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(cfg["base_model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, device_map="cuda").eval()
print("loaded", cfg["base_model"])

def model_for_phase(ckpt_path):
    """Swaps the LoRA adapter in place; base weights load once."""
    global base_model
    if isinstance(base_model, PeftModel):
        base_model = base_model.unload()
    if ckpt_path is not None:
        base_model = PeftModel.from_pretrained(base_model, ckpt_path).eval()
    return base_model

In [ ]:
# Llama-2's tokenizer ships no chat template, so eval.py's use_ct=True would silently
# fall back to a raw string the model was never tuned on. Set the tulu format (run_sft.py's
# _concat_messages) and verify byte-for-byte against a prompt the original eval stored.
TULU_CHAT_TEMPLATE = (
    "{{ bos_token }} "          # trailing space: reproduces the sentencepiece prefix
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)
tok.chat_template = TULU_CHAT_TEMPLATE

import sys
sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

REF = "results/cpi-results/tulu-results/checkpoint_step3150_metrics.json"
ref_row = next(r for r in json.load(open(REF, encoding="utf-8"))["per_item"]
               if r["item_id"] == "cap_0013")
ours = tok.decode(cpi_eval.build_score_prompt_ids(
    tok, ref_row["question"], ref_row["context"], True), skip_special_tokens=False)

assert ours == ref_row["score_prompt_text"], f"PROMPT MISMATCH\nours  : {ours!r}\nstored: {ref_row['score_prompt_text']!r}"
print("prompt verified -- byte-identical to the recorded eval")

### Items and the reference results

`passed` (the Layer-0 filter) and the stored `delta` come from the committed per-checkpoint results, so we measure exactly the item set the original trajectory used -- and get a free correctness check by comparing our recomputed `delta` against the stored one.

In [ ]:
items = {it["item_id"]: it for it in cpi_eval.load_items("dataset/conflict_eval_unified.json")}

# (run, phase, step, path to the committed result for that step)
PHASES = [
    ("alpaca", "peak",     100,  "results/cpi-results/alpaca-results/checkpoint_step100_metrics.json"),
    ("alpaca", "trough",   800,  "results/cpi-results/alpaca-results/checkpoint_step800_metrics.json"),
    ("tulu",   "peak",     500,  "results/cpi-results/tulu-results/checkpoint_step500_metrics.json"),
    ("tulu",   "trough",   3150, "results/cpi-results/tulu-results/checkpoint_step3150_metrics.json"),
    ("tulu",   "recovery", 5000, "results/cpi-results/tulu-results/checkpoint_step5000_metrics.json"),
]

def reference(path):
    """{item_id: stored delta} for items that passed the Layer-0 filter."""
    rows = json.load(open(path, encoding="utf-8"))["per_item"]
    return {r["item_id"]: r["logprob"]["delta"] for r in rows
            if r.get("passed") and r.get("logprob") and "delta" in r["logprob"]}

for run, phase, step, path in PHASES:
    print(f"{run:7} {phase:9} step{step:<5} passed={len(reference(path))}")

### Measure

In [ ]:
@torch.no_grad()
def first_token_gap(model, score_ids, par_tok, cf_tok):
    """logit(cf first token) - logit(par first token) at the last prompt position.
    Same sign convention as eval.py's delta: > 0 favours the CONTEXT answer."""
    logits = model(score_ids.unsqueeze(0).to(model.device)).logits[0, -1, :]
    return (logits[cf_tok] - logits[par_tok]).item()


def first_tok(text):
    return tok(" " + text.strip(), add_special_tokens=False).input_ids[0]


rows = []
for run, phase, step, path in PHASES:
    stored = reference(path)
    model = model_for_phase(phase_path(run, phase))
    print(f"measuring {run}/{phase} ({len(stored)} items) ...", flush=True)

    for iid, stored_delta in stored.items():
        it = items[iid]
        score_ids = cpi_eval.build_score_prompt_ids(
            tok, it["question"], it["context"], True)
        d = cpi_eval.method_logprob(model, tok, score_ids,
                                    it["parametric_answer"], it["counterfactual_answer"])
        g = first_token_gap(model, score_ids,
                            first_tok(it["parametric_answer"]),
                            first_tok(it["counterfactual_answer"]))
        rows.append({"run": run, "phase": phase, "step": step, "item_id": iid,
                     "delta": d["delta"], "first_gap": g, "stored_delta": stored_delta})

json.dump(rows, open("src/mechanistic-analysis/first_token_gap.json", "w"), indent=2)
print(f"\nwrote src/mechanistic-analysis/first_token_gap.json ({len(rows)} rows)")

### Check we reproduce the original measurement

In [ ]:
import numpy as np

for run, phase, step, _ in PHASES:
    sub = [r for r in rows if r["run"] == run and r["phase"] == phase]
    diff = np.array([r["delta"] - r["stored_delta"] for r in sub])
    print(f"{run:7} {phase:9} n={len(sub):4}  max|ours-stored|={np.abs(diff).max():.4f}  "
          f"mean={diff.mean():+.4f}")
print("\n(near-zero = our delta reproduces the recorded eval; large = something is off)")

### Result: does first_gap track R_ctx where delta doesn't?

In [ ]:
R_CTX = {("alpaca", "peak"): 0.890, ("alpaca", "trough"): 0.789,
          ("tulu", "peak"): 0.955, ("tulu", "trough"): 0.776, ("tulu", "recovery"): 0.902}

print(f"{'run':8}{'phase':10}{'n':>5}{'mean delta':>12}{'mean first_gap':>16}{'R_ctx':>8}")
summary = {}
for run, phase, step, _ in PHASES:
    sub = [r for r in rows if r["run"] == run and r["phase"] == phase]
    md_, mg = np.mean([r["delta"] for r in sub]), np.mean([r["first_gap"] for r in sub])
    summary[(run, phase)] = (md_, mg)
    print(f"{run:8}{phase:10}{len(sub):5}{md_:12.3f}{mg:16.3f}{R_CTX[(run, phase)]:8.3f}")

# paired per-item shifts across each leg
print("\nper-item shifts (paired on items passing at both ends):")
for run, ep, lp in [("alpaca", "peak", "trough"),
                    ("tulu", "peak", "trough"), ("tulu", "trough", "recovery")]:
    e = {r["item_id"]: r for r in rows if r["run"] == run and r["phase"] == ep}
    l = {r["item_id"]: r for r in rows if r["run"] == run and r["phase"] == lp}
    common = sorted(set(e) & set(l))
    dd = np.array([l[i]["delta"] - e[i]["delta"] for i in common])
    dg = np.array([l[i]["first_gap"] - e[i]["first_gap"] for i in common])
    print(f"  {run} {ep}->{lp}  n={len(common)}")
    print(f"     delta     shift: mean={dd.mean():+.3f}  toward PAR: {(dd < 0).sum()}/{len(dd)} ({(dd < 0).mean():.0%})")
    print(f"     first_gap shift: mean={dg.mean():+.3f}  toward PAR: {(dg < 0).sum()}/{len(dg)} ({(dg < 0).mean():.0%})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, run in zip(axes, ["alpaca", "tulu"]):
    ph = [p for r, p, _, _ in PHASES if r == run]
    x = range(len(ph))
    ax.plot(x, [summary[(run, p)][0] for p in ph], "o-", label="mean delta (full answer)")
    ax.plot(x, [summary[(run, p)][1] for p in ph], "s-", label="mean first_gap")
    ax.set_xticks(list(x)); ax.set_xticklabels(ph)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_ylabel("favours CONTEXT  -->")
    ax2 = ax.twinx()
    ax2.plot(x, [R_CTX[(run, p)] for p in ph], "^--", color="crimson", label="R_ctx")
    ax2.set_ylabel("R_ctx", color="crimson")
    ax.set_title(run); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.savefig("src/mechanistic-analysis/first_token_gap.png", dpi=150)
plt.show()

### Reading it

The hypothesis predicts `first_gap` falls peak->trough and rises again at recovery,
**tracking the red R_ctx line**, while `delta` stays roughly flat.

- **If first_gap tracks R_ctx and delta doesn't** -- the inversion is localised to
  first-token commitment. That's a sharp claim: the model's answer *preference* barely
  moves, what moves is which token it commits to first. Mechanistic work then targets
  `first_gap` (which is exactly what direct logit attribution decomposes), on all ~400
  items rather than a selected flip-set.
- **If neither tracks R_ctx** -- the inversion isn't in the scored-answer logits at all,
  and the next place to look is the generation dynamics themselves (EOS/formatting,
  answer-span position), not the candidate-answer scores.
- **If both track it** -- then the earlier flat delta was an artefact of the flip-set
  selection, and the ordinary continuous delta is a fine target after all.

Download `first_token_gap.json` (and the PNG) and hand them back.